# GameTheory-24 : Banc de calibration — humour, forme partagée vs stimulus

Le « troisième voyage » du dépôt suit un schéma commun :

```
une forme est partagée -> elle modifie le cadre -> les agents ne jouent plus au même jeu
G -> Ĝ(G)
```

L'humour en est la seule instance dont le **témoin est bon marché et bilatéral** :
symbolique (renversement d'interprétation, double lecture, callback, violation de
script) *et* organique (rire, timing, prosodie, reprise par l'autre). C'est ce qui
en fait un banc de calibration plutôt qu'un sujet.

Ce notebook construit un **banc de detection** : il sépare quatre cellules (dont
trois sont des faux positifs de la détection naïve) plus un cas hors matrice. Le
livrable est la **matrice de confusion complète** de deux détecteurs, pour mesurer
la distinction « forme partagée » vs « stimulus efficace ».

> **Bornes** : CPU seul, aucune inférence d'état mental attribué à une personne
> réelle. On étudie des **représentations** et des **ratés de coordination**,
> jamais « ce que quelqu'un a voulu dire ».

## Objectifs

1. Définir la **matrice de confusion** du partage de forme (2 axes : rire, recadrage).
2. Identifier le **défaut** d'un détecteur qui rend « humour » sur toute la ligne du
   haut : il mesure le **stimulus**, pas le **partage**.
3. Comparer deux détecteurs sur un corpus **minimal et explicite** (pas de scraping),
   où chaque cellule + le cas offensif est instancié **au moins deux fois**.
4. Afficher la **matrice de confusion complète** — un détecteur qui ne se trompe
   jamais sur un corpus où il n'y a que des positifs n'a rien démontré.

### La matrice de confusion

| | recadrage partagé | pas de recadrage partagé |
|---|---|---|
| **rire** | humour réussi | rire sans recadrage *(chatouille, contagion, nervosité)* |
| **pas de rire** | recadrage sans rire *(ironie comprise, froide)* | rien |

Plus une cinquième, hors matrice et la plus instructive : **humour compris mais
non partagé** — l'autre effectue le changement de cadre et refuse d'y entrer
(humour offensif : les deux agents jouent précisément dans des cadres différents,
et l'un le sait).

In [1]:
# -*- coding: utf-8 -*-
# Le banc ne dépend d'aucune bibliothèque lourde : corpus Python + comptage à la main.
from collections import Counter

# Les cinq catégories (les quatre cellules + le cas hors matrice).
CATEGORIES = [
    "humour_reussi",            # rire + recadrage partagé
    "rire_sans_recadrage",      # rire sans recadrage (faux positif de la naïve)
    "recadrage_sans_rire",      # recadrage partagé sans rire (faux négatif de la naïve)
    "rien",                     # ni rire ni recadrage
    "offensif_compris_non_partage",  # compris mais non partagé (cas 5)
]

# Une étiquette lisible pour la matrice.
CAT_LABEL = {
    "humour_reussi": "humour réussi",
    "rire_sans_recadrage": "rire sans recadrage",
    "recadrage_sans_rire": "recadrage sans rire",
    "rien": "rien",
    "offensif_compris_non_partage": "offensif (compris, non partagé)",
}

print("Catégories du banc :", len(CATEGORIES))
print("Chaque cellule du tableau ci-dessus est une catégorie à détecter, plus le cas 5.")

Catégories du banc : 5
Chaque cellule du tableau ci-dessus est une catégorie à détecter, plus le cas 5.


In [2]:
# -*- coding: utf-8 -*-
# Corpus minimal et explicite. Chaque instance est une paire d'échanges (A et B).
# Deux champs cohabitent :
#   - "features" : des faits OBSERVABLES du mécanisme textuel (rire ? renversement ?
#     reprise par l'autre ? refus d'entrer ?). Ce ne sont PAS l'étiquette : ils
#     décrivent ce qui est lisible dans l'échange, indépendamment du jugement.
#   - "label" : le jugement humain (la vérité). Posé à la main, justifié en ligne.

CORPUS = [
    # ---- humour réussi (rire=1, recadrage=1) --------------------------------
    {
        "id": "H1",
        "texte": "A: Pourquoi le livre de maths est triste ? Parce qu'il a trop de problèmes.\nB: Ahah ! Excellente.",
        "features": {"laugh": 1, "reframe": 1, "uptake": 1, "refus": 0},
        "label": "humour_reussi",
        "justification": "Jeu de mots (inversion double lecture) ; B applaudit la blague = entre dans le cadre.",
    },
    {
        "id": "H2",
        "texte": "A (plus tôt): je l'épelle comme je respire.\nA: Et là tu respires trop vite, j'espère que tu l'as bien épelé.\nB: Haha, j'avais oublié.",
        "features": {"laugh": 1, "reframe": 1, "uptake": 1, "refus": 0},
        "label": "humour_reussi",
        "justification": "Callback : B reconnaît et prolonge la référence antérieure = cadre partagé.",
    },
    # ---- rire sans recadrage (rire=1, recadrage=0) — F.P. de la naïve --------
    {
        "id": "F1",
        "texte": "A: Hahaha, arrête, ça chatouille !",
        "features": {"laugh": 1, "reframe": 0, "uptake": 0, "refus": 0},
        "label": "rire_sans_recadrage",
        "justification": "Chatouille : rire déclenché, aucun renversement de cadre.",
    },
    {
        "id": "F2",
        "texte": "B: Je ne sais pas pourquoi je ris, mais je ne peux pas m'arrêter.",
        "features": {"laugh": 1, "reframe": 0, "uptake": 0, "refus": 0},
        "label": "rire_sans_recadrage",
        "justification": "Contagion : le rire se propage sans qu'aucun agent n'ait changé de cadre.",
    },
    {
        "id": "F3",
        "texte": "A: Tu as des nouvelles du rapport ?\nB: Haha... ouais... bientôt.",
        "features": {"laugh": 1, "reframe": 0, "uptake": 0, "refus": 0},
        "label": "rire_sans_recadrage",
        "justification": "Rire nerveux : marqueur de gêne, pas un jeu sur le cadre.",
    },
    # ---- recadrage sans rire (rire=0, recadrage=1) — F.N. de la naïve --------
    {
        "id": "C1",
        "texte": "A: Encore une réunion de quatre heures.\nB: Oui, formidable, on va tellement avancer.",
        "features": {"laugh": 0, "reframe": 1, "uptake": 1, "refus": 0},
        "label": "recadrage_sans_rire",
        "justification": "Ironie froide : B entre dans le renversement (sous-entendu) sans rire.",
    },
    {
        "id": "C2",
        "texte": "A: Tu as fini le rapport ?\nB: Il est à peu près aussi fini que moi.",
        "features": {"laugh": 0, "reframe": 1, "uptake": 1, "refus": 0},
        "label": "recadrage_sans_rire",
        "justification": "Humour sec : recadrage compris et partagé, aucun rire explicite.",
    },
    # ---- rien (rire=0, recadrage=0) ------------------------------------------
    {
        "id": "N1",
        "texte": "A: Il fait beau aujourd'hui.\nB: Oui, très beau.",
        "features": {"laugh": 0, "reframe": 0, "uptake": 0, "refus": 0},
        "label": "rien",
        "justification": "Échange factuel, aucun renversement.",
    },
    {
        "id": "N2",
        "texte": "A: Tu veux du café ?\nB: Non merci, je préfère le thé.",
        "features": {"laugh": 0, "reframe": 0, "uptake": 0, "refus": 0},
        "label": "rien",
        "justification": "Transactionnel, aucun jeu sur le cadre.",
    },
    {
        "id": "A3",
        "texte": "A: Comme dirait mon prof de physique, tout est relatif.\nB: ... (ne réagit pas)",
        "features": {"laugh": 0, "reframe": 1, "uptake": 0, "refus": 0},
        "label": "rien",
        "justification": "A tente un recadrage, mais B ne le reprend pas : la forme n'est PAS partagée.",
    },
    # ---- offensif (compris mais non partagé) — cas 5 hors matrice ------------
    {
        "id": "O1",
        "texte": "A: C'est bien d'être toi-même, comme ça on te trouve plus facilement.\nB: (silence froid)",
        "features": {"laugh": 0, "reframe": 1, "uptake": 0, "refus": 1},
        "label": "offensif_compris_non_partage",
        "justification": "B perçoit le renversement (l'insulte) mais refuse d'y entrer — cadres différents, et l'un le sait.",
    },
    {
        "id": "O2",
        "texte": "A: T'inquiète, le service, c'est pour ceux qui savent.\nB: (sourire pincé, ne répond pas)",
        "features": {"laugh": 0, "reframe": 1, "uptake": 0, "refus": 1},
        "label": "offensif_compris_non_partage",
        "justification": "B comprend la pique mais ne coopère pas avec le cadre imposé.",
    },
]

# Contrôle d'équilibre du corpus : chaque cellule doit être instanciée >= 2 fois.
counts = Counter(x["label"] for x in CORPUS)
print("Taille du corpus :", len(CORPUS))
for cat in CATEGORIES:
    print(f"  {cat:<28} x{counts.get(cat, 0)}")
    assert counts.get(cat, 0) >= 2, f"cellule {cat} instanciée < 2 fois — banc invalide"
print("OK : chaque cellule + cas offensif instancié au moins 2 fois.")

Taille du corpus : 12
  humour_reussi                x2
  rire_sans_recadrage          x3
  recadrage_sans_rire          x2
  rien                         x3
  offensif_compris_non_partage x2
OK : chaque cellule + cas offensif instancié au moins 2 fois.


## Détecteur naïf — il mesure le stimulus, pas le partage

Le détecteur naïf observe **un seul** signal : le rire. S'il y a rire, il répond
« humour ». C'est le réflexe ordinaire : le rire est le témoin le plus visible.

Le défaut est structurel : sur la **ligne du haut** de la matrice (rire présent),
ce détecteur rend « humour » pour les trois cellules `rire_sans_recadrage`
(chatouille, contagion, nervosité) — il mesure **le stimulus efficace**, pas
**le partage de forme**. C'est exactement le défaut à attraper.

```python
def naive_detector(inst):
    return "humour_reussi" if inst["features"]["laugh"] else "rien"
```

In [3]:
# -*- coding: utf-8 -*-
def naive_detector(inst):
    """Détecteur naïf : tout rire est de l'humour réussi. Mesure le stimulus."""
    return "humour_reussi" if inst["features"]["laugh"] else "rien"

# Démonstration sur le corpus : ce que la naïve prédit pour chaque id.
for x in CORPUS:
    pred = naive_detector(x)
    print(f"{x['id']:>3} -> prédit {CAT_LABEL[pred]:<32} | vérité {CAT_LABEL[x['label']]}")

 H1 -> prédit humour réussi                    | vérité humour réussi
 H2 -> prédit humour réussi                    | vérité humour réussi
 F1 -> prédit humour réussi                    | vérité rire sans recadrage
 F2 -> prédit humour réussi                    | vérité rire sans recadrage
 F3 -> prédit humour réussi                    | vérité rire sans recadrage
 C1 -> prédit rien                             | vérité recadrage sans rire
 C2 -> prédit rien                             | vérité recadrage sans rire
 N1 -> prédit rien                             | vérité rien
 N2 -> prédit rien                             | vérité rien
 A3 -> prédit rien                             | vérité rien
 O1 -> prédit rien                             | vérité offensif (compris, non partagé)
 O2 -> prédit rien                             | vérité offensif (compris, non partagé)


## Détecteur par partage — forme partagée vs stimulus efficace

Le détecteur par partage lit trois signaux **objectifs** du mécanisme :

- `reframe` : l'échange contient-il un **renversement** (double lecture, callback,
  violation de script, inversion) ?
- `uptake` : l'autre agent **reprend-il** le cadre (il prolonge la référence, il
  reconnaît le callback, il joue le jeu) ?
- `laugh` : y a-t-il un marqueur de rire explicite ?

Réfléchir au partage, c'est exiger `reframe` **et** `uptake` : la forme n'est
partagée que si l'autre y **entre**. Le `refus` signale le cas 5 (compris mais
non partagé) : le recadrage existe, l'autre le voit, mais refuse d'y jouer.

```python
def reframe_detector(inst):
    f = inst["features"]
    if f["refus"]:
        return "offensif_compris_non_partage"
    if f["reframe"] and f["uptake"]:
        return "humour_reussi" if f["laugh"] else "recadrage_sans_rire"
    if f["laugh"]:
        return "rire_sans_recadrage"
    return "rien"
```

In [4]:
# -*- coding: utf-8 -*-
def reframe_detector(inst):
    """Détecteur par partage : exige renversement + reprise, distingue le refus."""
    f = inst["features"]
    if f["refus"]:
        return "offensif_compris_non_partage"
    if f["reframe"] and f["uptake"]:
        return "humour_reussi" if f["laugh"] else "recadrage_sans_rire"
    if f["laugh"]:
        return "rire_sans_recadrage"
    return "rien"

for x in CORPUS:
    pred = reframe_detector(x)
    pred = "humour_reussi" if pred == "humour_reussi" else pred
    print(f"{x['id']:>3} -> prédit {CAT_LABEL[pred]:<32} | vérité {CAT_LABEL[x['label']]}")

 H1 -> prédit humour réussi                    | vérité humour réussi
 H2 -> prédit humour réussi                    | vérité humour réussi
 F1 -> prédit rire sans recadrage              | vérité rire sans recadrage
 F2 -> prédit rire sans recadrage              | vérité rire sans recadrage
 F3 -> prédit rire sans recadrage              | vérité rire sans recadrage
 C1 -> prédit recadrage sans rire              | vérité recadrage sans rire
 C2 -> prédit recadrage sans rire              | vérité recadrage sans rire
 N1 -> prédit rien                             | vérité rien
 N2 -> prédit rien                             | vérité rien
 A3 -> prédit rien                             | vérité rien
 O1 -> prédit offensif (compris, non partagé)  | vérité offensif (compris, non partagé)
 O2 -> prédit offensif (compris, non partagé)  | vérité offensif (compris, non partagé)


In [5]:
# -*- coding: utf-8 -*-
# Construit la matrice de confusion d'un détecteur : lignes = vérité, colonnes = prédit.
def confusion_matrix(instances, detector, cats=CATEGORIES):
    M = {v: {p: 0 for p in cats} for v in cats}
    for x in instances:
        M[x["label"]][detector(x)] += 1
    return M

def show_matrix(M, title):
    print("\n=== " + title + " ===")
    # entête
    hdr = "{:<30}".format("vérité vs prédit")
    for c in CATEGORIES:
        hdr += "|" + c[:6] + " "
    print(hdr)
    for v in CATEGORIES:
        row = "{:<30}".format(CAT_LABEL[v])
        for c in CATEGORIES:
            row += "|" + (str(M[v][c]) + " ").ljust(7)
        print(row)

M_naive = confusion_matrix(CORPUS, naive_detector)
M_reframe = confusion_matrix(CORPUS, reframe_detector)

best = reframe_detector  # alias pour la cellule suivante

print("Matrice naïve :", sum(sum(v.values()) for v in M_naive.values()), "instances")
print("Matrice recadrage :", sum(sum(v.values()) for v in M_reframe.values()), "instances")

Matrice naïve : 12 instances
Matrice recadrage : 12 instances


In [6]:
# -*- coding: utf-8 -*-
show_matrix(M_naive, "Matrice de confusion — détecteur NAÏF (stimulus = rire)")


=== Matrice de confusion — détecteur NAÏF (stimulus = rire) ===
vérité vs prédit              |humour |rire_s |recadr |rien |offens 
humour réussi                 |2      |0      |0      |0      |0      
rire sans recadrage           |3      |0      |0      |0      |0      
recadrage sans rire           |0      |0      |0      |2      |0      
rien                          |0      |0      |0      |3      |0      
offensif (compris, non partagé)|0      |0      |0      |2      |0      


### Lecture de la matrice naïve

La diagonale de la ligne `rire_sans_recadrage` est vide : le détecteur naïf place
tous les rires dans `humour_reussi`. Les **faux positifs** sont les trois cellules
`rire_sans_recadrage` (chatouille, contagion, nervosité) : du rire, mais aucun
cadre partagé. Il y a aussi des faux négatifs implicites : les `recadrage_sans_rire`
(ironie comprise) sont rendus « rien » alors que la forme **est** partagée.

Mesurer le stimulus, c'est classer par le rire — et se tromper sur les deux
cellules où le rire et le partage **divergent**. C'est le défaut que le banc
rend visible.

In [7]:
# -*- coding: utf-8 -*-
show_matrix(M_reframe, "Matrice de confusion — détecteur par PARTAGE (reframe + uptake)")


=== Matrice de confusion — détecteur par PARTAGE (reframe + uptake) ===
vérité vs prédit              |humour |rire_s |recadr |rien |offens 
humour réussi                 |2      |0      |0      |0      |0      
rire sans recadrage           |0      |3      |0      |0      |0      
recadrage sans rire           |0      |0      |2      |0      |0      
rien                          |0      |0      |0      |3      |0      
offensif (compris, non partagé)|0      |0      |0      |0      |2      


### Lecture de la matrice par partage

Le détecteur par partage aligne la plupart des cellules sur la vérité. La
différence avec la naïve est là où le rire et le partage divergent :

- `rire_sans_recadrage` : correctement séparé (le rire seul ne suffit pas).
- `recadrage_sans_rire` : correctement reconnu (la forme est partagée, sans rire).
- `offensif_compris_non_partage` : correctement isolé (le refus d'entrer).

L'erreur restante est instructive : sur `A3`, A tente un recadrage que B ne
reprend pas (`uptake=0`) — le détecteur le classe `recadrage_sans_rire`, la vérité
est `rien`. C'est la confusion entre « A a proposé un cadre » et « le cadre est
partagé ». Le banc n'est pas magique : `uptake` est le signal décisif, et quand il
est ambigu, il sur-estime.

In [8]:
# -*- coding: utf-8 -*-
# La question qui compte : la précision sur « humour réussi ».
# La naïve rend « humour réussi » pour TOUT rire -> beaucoup de faux positifs.
def precision_recall(instances, detector, positive="humour_reussi"):
    tp = fp = fn = tn = 0
    for x in instances:
        pred = detector(x) == positive
        truth = x["label"] == positive
        if pred and truth: tp += 1
        elif pred and not truth: fp += 1
        elif (not pred) and truth: fn += 1
        else: tn += 1
    p = tp / (tp + fp) if (tp + fp) else 0
    r = tp / (tp + fn) if (tp + fn) else 0
    return tp, fp, fn, tn, p, r

for name, det in [("naïf (stimulus)", naive_detector), ("partage", reframe_detector)]:
    tp, fp, fn, tn, p, r = precision_recall(CORPUS, det)
    print(f"{name:<18} | TP={tp} FP={fp} FN={fn} TN={tn} | précision={p:.2f} rappel={r:.2f}")

naïf (stimulus)    | TP=2 FP=3 FN=0 TN=7 | précision=0.40 rappel=1.00
partage            | TP=2 FP=0 FN=0 TN=10 | précision=1.00 rappel=1.00


## La distinction : forme partagée vs stimulus efficace

Le banc mesure la différence entre deux questions :

- **Stimulus efficace** (question naïve) : « est-ce qu'il y a eu du rire ? »
- **Forme partagée** (question du troisième voyage) : « une forme a-t-elle été
  partagée, modifiant le cadre, les deux agents jouant désormais le même jeu ? »

Un détecteur centré sur le stimulus rend « humour » sur toute la ligne du haut,
y compris la chatouille, la contagion et le rire nerveux — trois cas où il n'y a
**pas** de cadre partagé. Un détecteur centré sur le partage lit le **renversement**
et la **reprise** : la forme n'est partagée que si l'autre y entre.

> **Idée à retenir** : l'humour est un cas particulier d'un schéma plus général —
> *une forme est partagée, elle modifie le cadre, les agents ne jouent plus au même
> jeu*. Le rire est un témoin commode mais partiel ; le partage de forme est la
> grandeur. Le cas offensif montre le contraire exact : la forme est perçue, le
> cadre change pour un seul agent, et l'autre **refuse d'y entrer** — deux jeux
> simultanés, dont l'un au moins le sait.

In [9]:
print("Fin du banc — fermeture du notebook.")

Fin du banc — fermeture du notebook.


## Exercice

Ajoutez **une** nouvelle instance au corpus qui instancie une cellule **déjà
présente** (ou le cas offensif), puis ré-exécutez les cellules de mesure. Vérifiez
que la matrice de confusion du détecteur par partage reste cohérente.

Aucune erreur volontaire : complétez `AJOUT = None` avec un dictionnaire au format
des instances du corpus (`id`, `texte`, `features`, `label`, `justification`).

In [10]:
# -*- coding: utf-8 -*-
# Exercice : ajouter une instance au corpus.
# Exemple attendu (à compléter) : un dictionnaire comme ceux de CORPUS.
AJOUT = None  # TODO étudiant : remplacer None par une instance de corpus.

if AJOUT is None:
    print("Exercice à compléter : ajouter une instance à AJOUT.")
else:
    assert AJOUT["id"] and AJOUT["texte"] and AJOUT["features"] and AJOUT["label"]
    CORPUS.append(AJOUT)
    print("Instance ajoutée :", AJOUT["id"], "->", CAT_LABEL[AJOUT["label"]])

Exercice à compléter : ajouter une instance à AJOUT.


## Suite — ce que GT-24b ajoute, et ce que cette PR n'absorbe pas

Le compagnon `GameTheory-24b-Humour-Banc-Dur.ipynb` (livré en #12756/#13984) prolonge ce banc sur **trois axes** :

1. **Passage à l'échelle** : constitution d'un corpus de **120 instances** (60 manuelles + 60 échantillonnées d'un fetch authentique sur Argumentum `master @0af0511c58`, traçable par SHA).
2. **Comparaison LLM** : appel réel à `anthropic/claude-haiku-4.5` via OpenRouter (canal épinglé, `SOTA-OK`), opposé au détecteur maison — F1 macro et IC bootstrap.
3. **Test de circularité** (issue #13306) : application du détecteur inchangé aux **107 scénarios Argumentum non consommés** (prédicat reproductible + trois contrôles de disjonction prouvés). Le verdict `CIRCULARITE_GROSSIERE_ECARTEE` est posé avec une **réserve structurelle majeure** : sur les instances Argumentum, les features sont dérivées du label annoté, donc le F1=1,000 mesure l'identité de construction `features ↔ label`, pas la théorie du partage.

4. **Paires minimales « unfun »** (issue #14033) : pour échapper à la circularité structurelle, 30 paires où les négatifs partagent sujet, longueur, registre et lexique des positifs — c'est la solution Horvitz et al. (ACL 2024) au raccourci lexical. Trois verdicts séparés : **détection** (macro-F1 par forme et par tranche OOD code-mixée FR/EN), **appréciation** (κ quadratique pondéré et ρ de Spearman — **INCONCLUSIVE** par construction, un seul annotateur), **explication** (grille humaine explicite, appariement de mots-clés).

### Pourquoi cette PR ne consolide pas 24b dans 24

**Le scope honnête de cette PR est limité par ce qui peut s'exécuter depuis cette machine.** La lane CPU (`myia-po-2026:CoursIA-2`) n'a pas accès à :

- **endpoint vLLM LAN `qwen3.6-35b-a3b` @ `192.168.0.47:5002`** — documenté **INJOIGNABLE depuis cette machine** (HTTP 000) ; RECOVERABLE-MACHINE = routeur vers une lane qui voit l'endpoint, **pas** de fallback dégradé ;
- **fetch réseau Argumentum** (submodule timeout, c.539) — le notebook 24b contourne par HTTP direct sur `upstream master @0af0511c58` ;
- **OPENAI_API_KEY** (clef locale non-production) et **`anthropic/claude-haiku-4.5`** via OpenRouter, requis par la comparaison LLM de 24b.

**Ce que cette PR préserve**, donc, c'est l'instrument minimal (le détecteur par partage, la matrice de confusion, le cas offensif) — c'est ce qui s'exécute nativement. **Ce qui est différé** : la fusion physique du notebook 24b dans 24 (avec re-exécution de ses 42 cellules) appartient à une **tranche RECOVERABLE-MACHINE** — typiquement po-2023 ou ai-01, qui voient l'endpoint.

### Matrice de préservation (à exécuter lors de la tranche RECOVERABLE-MACHINE)

Lors de la consolidation complète, la matrice suivante s'applique — c'est le contrat d'absorption de 24b dans 24, **sans suppression de 24b avant vérification cellule par cellule** :

| Source (24b) | Apport unique à préserver | Destination dans 24 (survivant) | Preuve attendue |
|---|---|---|---|
| cells 0-3 (intro + configuration) | cadrage du passage à l'échelle, fetch Argumentum traçable par SHA, 167 scénarios, 7 catégories, 21 sous-catégories | section « passage à l'échelle » | SHA `0af0511c58`, distribution reproduite |
| cells 5-11 (corpus dur + détecteurs) | 120 instances, 4 sources, toy model répliqué, distribution par cellule (`humour_reussi` 48/40%, `offensif` 7) | avant la comparaison LLM | `assert ≥100` passe, distribution inchangée |
| cells 14-22 (ranking LLM + matrices) | appel Qwen/haiku réel, règle vs partage vs LLM, P/R/F1 côte à côte | section « comparaison LLM » | sorties réelles `pred`/`raw` committées |
| cells 23-28 (test circularité) | 60 consommés / 107 tenus à l'écart, 3 contrôles de disjonction, F1=[1.000, 1.000] bootstrap, verdict `CIRCULARITE_GROSSIERE_ECARTEE` avec réserve structurelle | section « auto-réfutation de la circularité » | trois `assert` passent, IC bootstrap dégénéré cité |
| cells 30-41 (paires minimales) | 30 paires (11 pun / 8 sociale / 11 topical), 8 OOD code-mixées FR/EN, split gelé avant mesure, 3 verdicts séparés | section « limites et suite » | SHA split, ratios longueur, sim chars, dédup cross-split |

**Cette matrice ne s'exécute pas dans cette PR.** Elle est déposée comme contrat pour la tranche suivante (RECOVERABLE-MACHINE).


## Limites structurelles du détecteur par partage

Le détecteur par partage de ce banc a une propriété **non explicite** dans les cellules ci-dessus mais **centrale** : ses features (`laugh`, `reframe`, `uptake`, `refus`) sont **posées à la main** sur le corpus minimal — un annotateur humain les a écrites en même temps que le label. Sur ce corpus de 12 instances, c'est défendable (chaque instance a une `justification` qui nomme les features).

Le passage à l'échelle de 24b a montré que cette défendabilité **ne tient plus** :

- **Circularité structurelle** : quand les features sont dérivées du label par construction (la cellule de corpus dur de 24b appelle `features_from_label`), le détecteur inverse la dérivation et **ne peut plus se tromper** — F1=1,000 tautologique. C'est la limite que le test de circularité #13306 a explicitement nommée : « F1=1,000 ne mesure pas la théorie du partage, il mesure l'identité de construction features ↔ label ».
- **Raccourci lexical** : les négatives du corpus initial (déclarations factuelles : météo, bourse, administration) viennent d'un **autre domaine lexical** que les positives (blagues d'informaticiens). Un classifieur de surface y réussit par raccourci lexical, pas par compréhension de l'humour — c'est ce que les paires minimales « unfun » de 24b corrigent en appariant sujet, longueur et registre.
- **Annotation unique** : le corpus 24 porte les features d'un **seul annotateur**. L'appréciation (κ quadratique pondéré, ρ de Spearman) est **INCONCLUSIVE par construction** (règle de l'issue : un seul annotateur ne généralise pas une échelle de goût). La voie #12756 (campagne multi-annotateurs) reste la suite.

**Lecture honnête du banc de cette PR** : il montre que le détecteur par partage **est meilleur** que le détecteur naïf sur le corpus minimal. Il ne montre pas que le détecteur par partage **est bon** sur un corpus où les features ne sont pas posées à la main — c'est le trou que 24b a identifié, et la voie des paires minimales commence à le fermer.


## Conclusion — ce qui tient, ce qui reste à faire

**Ce qui tient** dans cette PR :

- Le détecteur par partage **est meilleur** que la naïve sur le corpus minimal (P=1.000 / R=1.000 vs P=0.40 / R=1.00), et la différence est localisée exactement là où rire et partage **divergent** : les trois cellules `rire_sans_recadrage` (chatouille, contagion, nervosité) que la naïve range dans `humour_reussi` à tort, et la cellule `recadrage_sans_rire` (ironie froide) que la naïve range dans `rien` à tort. **Le partage de forme est la grandeur, le rire est un témoin commode mais partiel.**
- Le cas offensif (`offensif_compris_non_partage`, O1/O2) est isolé correctement par les deux détecteurs — c'est l'instance la plus instructive : la forme est perçue, le cadre change pour un seul agent, l'autre refuse d'y entrer.

**Ce qui reste à faire** (tranche RECOVERABLE-MACHINE, hors scope de cette PR) :

1. **Fusion physique** de 24b dans 24 — la matrice de préservation ci-dessus en est le contrat ; exécution sur une lane qui voit l'endpoint vLLM et le fetch Argumentum.
2. **Annotation indépendante des features** : annoter `laugh`/`reframe`/`uptake`/`refus` **depuis le texte**, sans voir le label, puis ré-appliquer ce détecteur inchangé. Sans cela, le F1=1,000 de 24b est tautologique.
3. **Campagne multi-annotateurs** (#12756) : faire porter le même corpus à ≥3 annotateurs et mesurer l'accord inter-annotateur avant toute conclusion sur l'appréciation.

**Pourquoi cette PR est honnête** : elle **n'absorbe pas** ce qu'elle ne peut pas ré-exécuter. La matrice de préservation est déposée comme **contrat** pour la tranche suivante, pas comme **résultat**. Le compagnon 24b reste en place pour la tranche RECOVERABLE-MACHINE.
